# Weeks 4–5 · Data Manipulation with Pandas

**Programming for Data Science (606315)** · University of Petra

Pandas puts labels on NumPy arrays. A `DataFrame` is a table whose columns are typed NumPy arrays
sharing a row index; everything you learned about masks, broadcasting and `axis` still applies.
These two weeks take you from loading a CSV to a complete exploratory data analysis (EDA) —
exactly what Project Stage 1 asks for.

| Section | Topic |
|---|---|
| 1 | Series and DataFrame — the two objects |
| 2 | Loading data and first inspection |
| 3 | Selecting rows and columns: `[]`, `.loc`, `.iloc`, masks |
| 4 | Cleaning: types, whitespace, case, duplicates |
| 5 | Missing values |
| 6 | New columns, `apply`, `map`, string and category accessors |
| 7 | Sorting and ranking |
| 8 | **GroupBy** — split, apply, combine |
| 9 | Merge, join, concat |
| 10 | Pivot tables and reshaping |
| 11 | Time series |
| 12 | Quick plots |
| 13 | A complete EDA walk-through (your Stage 1 template) |

> In VS Code, click the **Data Wrangler** icon that appears above any DataFrame output to explore it interactively.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

## 1 · Series and DataFrame

A `Series` is a 1-D labelled array; a `DataFrame` is a dict of Series sharing an index.

In [ ]:
s = pd.Series([3.4, 2.1, 3.9], index=["Ahmad", "Sara", "Omar"], name="gpa")
print(s, s.index, s.values, s.dtype, sep="\n\n")
print("\nlabel access:", s["Sara"], "  position access:", s.iloc[1])

In [ ]:
df = pd.DataFrame({
    "name": ["Ahmad", "Sara", "Omar", "Lina"],
    "major": ["DS", "CS", "DS", "AI"],
    "gpa": [3.4, 2.1, 3.9, 3.2],
    "year": [2, 4, 1, 3],
})
df

In [ ]:
print(df.shape, df.columns.tolist(), df.dtypes.to_dict(), sep="\n")
df["gpa"]            # a column is a Series

## 2 · Loading data and first inspection

Three files generated for this course live in `data/generated/`. Always run these five commands on a new dataset
**before** doing anything else: `head`, `info`, `describe`, `isna().sum()`, `nunique`.

In [ ]:
students = pd.read_csv(DATA / "students.csv")
students.head()

In [ ]:
students.info()

`gpa` loaded as **object**, not float — a sign of dirty values. `describe()` on numeric columns, and `describe(include="object")` on text:

In [ ]:
students.describe()

In [ ]:
students.describe(include="object").T

In [ ]:
print(students.isna().sum(), "\n")
print(students.nunique())

## 3 · Selecting data

| Want | Use |
|---|---|
| one column | `df["col"]` |
| several columns | `df[["a", "b"]]` |
| rows by **label** (and columns) | `df.loc[rows, cols]` |
| rows by **position** | `df.iloc[rows, cols]` |
| rows by condition | `df[mask]` or `df.loc[mask, cols]` |
| SQL-style string filter | `df.query("gpa > 3 and year == 4")` |

In [ ]:
students[["name", "major"]].head(3)

In [ ]:
print(students.loc[0, "name"], "|", students.iloc[0, 1])    # same cell: label vs position
students.iloc[2:5, :3]

In [ ]:
seniors_amman = students[(students["year"] == 4) & (students["city"] == "Amman")]
print(len(seniors_amman))
seniors_amman.head(3)

In [ ]:
students.query("year == 4 and city == 'Amman'").equals(seniors_amman)

## 4 · Cleaning

Our `students.csv` was deliberately dirtied: `"unknown"` and blank GPAs, stray whitespace in `major`, inconsistent case in `city`, and duplicate rows. This is the normal state of real data.

In [ ]:
print(students["major"].unique())
print(students["city"].unique())
print("duplicates:", students.duplicated().sum())

In [ ]:
clean = students.copy()
clean["major"] = clean["major"].str.strip()
clean["city"]  = clean["city"].str.title()
clean["gpa"]   = pd.to_numeric(clean["gpa"], errors="coerce")   # "unknown", "" → NaN
clean = clean.drop_duplicates().reset_index(drop=True)

print(clean["major"].unique(), clean["city"].unique())
print(clean.dtypes["gpa"], "| rows:", len(students), "→", len(clean))

Chaining with `.assign` and `.pipe` gives the same result in one readable expression — this style is common in production code.

In [ ]:
clean2 = (students
          .assign(major=lambda d: d["major"].str.strip(),
                  city=lambda d: d["city"].str.title(),
                  gpa=lambda d: pd.to_numeric(d["gpa"], errors="coerce"))
          .drop_duplicates()
          .reset_index(drop=True))
clean2.equals(clean)

## 5 · Missing values

`NaN` propagates through arithmetic and is *skipped* by aggregations. Decide, column by column, whether to **drop**, **fill**, or **flag**.

In [ ]:
print(clean["gpa"].isna().sum(), "missing GPAs")
print("mean skips NaN:", clean["gpa"].mean().round(3))

dropped = clean.dropna(subset=["gpa"])                        # option 1: drop
filled  = clean.fillna({"gpa": clean["gpa"].median()})        # option 2: fill with a statistic
by_major = clean["gpa"].fillna(clean.groupby("major")["gpa"].transform("median"))  # option 3: group-wise fill
flagged = clean.assign(gpa_missing=clean["gpa"].isna())       # option 4: keep the information

print(len(dropped), filled["gpa"].isna().sum(), by_major.isna().sum(), flagged["gpa_missing"].sum())

## 6 · New columns, `apply`, `map`, and accessors

Prefer vectorised column arithmetic; fall back to `map` for value lookups and `apply` for row-wise logic you cannot vectorise.

In [ ]:
clean["gpa_pct"] = clean["gpa"] / 4 * 100                                     # vectorised
clean["standing"] = pd.cut(clean["gpa"], bins=[0, 2, 3.5, 4],
                           labels=["probation", "good", "honours"])           # binning
clean["major_code"] = clean["major"].map({"Data Science": "DS", "Computer Science": "CS",
                                          "Cybersecurity": "CY", "AI": "AI", "Software Eng": "SE"})
clean["first_name"] = clean["name"].str.split().str[0]                        # .str accessor
clean["domain"] = clean["email"].str.extract(r"@(.+)$")
clean["load"] = clean.apply(lambda r: "heavy" if r["credits"] / max(r["year"], 1) > 30 else "normal", axis=1)
clean.head()

In [ ]:
clean["standing"].value_counts(dropna=False)

### The `category` dtype

Text columns with few distinct values should be categorical: less memory, faster groupby, and an explicit order for sorting.

In [ ]:
before = clean["major"].memory_usage(deep=True)
clean["major"] = clean["major"].astype("category")
print(f"{before:,} → {clean['major'].memory_usage(deep=True):,} bytes")
print(clean["major"].cat.categories.tolist())

## 7 · Sorting and ranking

In [ ]:
clean.sort_values(["major", "gpa"], ascending=[True, False]).head(6)[["name", "major", "gpa"]]

In [ ]:
clean["rank_in_major"] = clean.groupby("major", observed=True)["gpa"].rank(ascending=False, method="min")
clean.nlargest(5, "gpa")[["name", "major", "gpa", "rank_in_major"]]

## 8 · GroupBy — split, apply, combine

`groupby` splits the frame by key, applies a function to each piece, and combines the results.
It is the Pandas equivalent of SQL `GROUP BY` and the heart of every summary table.

In [ ]:
clean.groupby("major", observed=True)["gpa"].mean().round(2)

In [ ]:
summary = (clean.groupby("major", observed=True)
                .agg(n=("student_id", "size"),
                     mean_gpa=("gpa", "mean"),
                     max_gpa=("gpa", "max"),
                     missing_gpa=("gpa", lambda s: s.isna().sum()),
                     cities=("city", "nunique"))
                .round(2)
                .sort_values("mean_gpa", ascending=False))
summary

In [ ]:
# two keys → MultiIndex; unstack turns the inner key into columns
clean.groupby(["major", "year"], observed=True)["gpa"].mean().unstack().round(2)

In [ ]:
# transform: a group statistic broadcast back to every row (for z-scores within group)
g = clean.groupby("major", observed=True)["gpa"]
clean["gpa_z_in_major"] = (clean["gpa"] - g.transform("mean")) / g.transform("std")
clean[["name", "major", "gpa", "gpa_z_in_major"]].head()

In [ ]:
# filter: keep only groups satisfying a condition
big_majors = clean.groupby("major", observed=True).filter(lambda d: len(d) >= 40)
big_majors["major"].value_counts()

## 9 · Merge, join, concat

`enrollments.csv` has one row per (student, course). Joining it to `students` is a **left merge** on `student_id`;
joining course titles from `courses.json` is another.

In [ ]:
enr = pd.read_csv(DATA / "enrollments.csv")
courses = pd.read_json(DATA / "courses.json")
print(enr.shape, courses.shape)
enr.head(3)

In [ ]:
full = (enr.merge(clean[["student_id", "name", "major"]], on="student_id", how="left")
           .merge(courses, left_on="course_code", right_on="code", how="left")
           .drop(columns="code"))
print(full.shape)
full.head()

In [ ]:
# how= matters. Validate with an indicator when you are not sure both sides match.
chk = enr.merge(clean[["student_id"]], on="student_id", how="outer", indicator=True)
chk["_merge"].value_counts()

In [ ]:
# concat stacks frames vertically (same columns) or horizontally (axis=1)
part1, part2 = clean.iloc[:100], clean.iloc[100:]
pd.concat([part1, part2]).shape == clean.shape

## 10 · Pivot tables and reshaping

A pivot table is a groupby with two keys laid out as rows × columns. `melt` is its inverse (wide → long).

In [ ]:
pt = full.pivot_table(index="major", columns="title", values="grade", aggfunc="mean", observed=True).round(1)
pt

In [ ]:
pt.reset_index().melt(id_vars="major", var_name="course", value_name="mean_grade").head()

In [ ]:
pd.crosstab(clean["major"], clean["standing"], margins=True)

## 11 · Time series

Parse dates on load, set the date as index, and the whole resample/rolling toolkit opens up.

In [ ]:
sales = pd.read_csv(DATA / "sales_2025.csv", parse_dates=["date"])
sales["revenue"] = sales["units"] * sales["unit_price"]
print(sales.dtypes["date"])
sales.head(3)

In [ ]:
daily = sales.groupby("date")["revenue"].sum()
monthly = daily.resample("ME").sum().round(0)          # 'ME' = month end
monthly

In [ ]:
# calendar features from the .dt accessor
sales["month"] = sales["date"].dt.month
sales["weekday"] = sales["date"].dt.day_name()
sales.groupby("weekday")["revenue"].mean().round(0).sort_values(ascending=False)

In [ ]:
rolling = daily.rolling(window=7).mean()          # 7-day moving average
pd.DataFrame({"daily": daily, "7d_avg": rolling}).iloc[5:12].round(0)

## 12 · Quick plots

`df.plot()` wraps Matplotlib. Good enough for EDA; you will polish the project charts later.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
clean["gpa"].plot.hist(bins=20, ax=axes[0], title="GPA distribution")
summary["mean_gpa"].plot.bar(ax=axes[1], title="Mean GPA by major", rot=30)
daily.rolling(7).mean().plot(ax=axes[2], title="Revenue, 7-day average")
plt.tight_layout()

## 13 · A complete EDA walk-through

This is the structure your **Project Stage 1** notebook should follow. Each heading is one cell of commentary plus the code that supports it.

1. **Load** and state shape, columns, dtypes.
2. **Quality audit**: missing values, duplicates, inconsistent categories, suspicious ranges.
3. **Clean**: fix what you found, and say what you did and why.
4. **Univariate**: distribution of each key variable (`describe`, `value_counts`, histogram).
5. **Bivariate / group**: `groupby`, `pivot_table`, `crosstab`, correlations.
6. **Findings**: 3–5 bullet points, each backed by a number from above.

In [ ]:
# 1. Load
raw = pd.read_csv(DATA / "students.csv")
print(raw.shape)

# 2. Audit
audit = pd.DataFrame({
    "dtype": raw.dtypes,
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "example": raw.iloc[0],
})
display(audit)
print("duplicates:", raw.duplicated().sum())
print("non-numeric gpa tokens:", raw.loc[pd.to_numeric(raw["gpa"], errors="coerce").isna(), "gpa"].unique())

In [ ]:
# 3. Clean
df = (raw.assign(major=lambda d: d["major"].str.strip(),
                 city=lambda d: d["city"].str.title(),
                 gpa=lambda d: pd.to_numeric(d["gpa"], errors="coerce"))
         .drop_duplicates()
         .reset_index(drop=True))
print(f"{len(raw)} → {len(df)} rows; {df['gpa'].isna().sum()} GPAs missing (kept as NaN)")

# 4. Univariate
display(df[["gpa", "credits", "year"]].describe().round(2))
display(df["major"].value_counts())

# 5. Bivariate
display(df.groupby("major")["gpa"].agg(["count", "mean", "std"]).round(2))
display(df[["gpa", "credits", "year"]].corr().round(2))

**6. Findings (example)**

* 205 raw rows contain 5 exact duplicates; after de-duplication there are 200 students.
* 4 % of GPAs were unparseable (`"unknown"`, blank) and are kept as missing rather than imputed, because they are few and the missingness looks random.
* Mean GPA is similar across majors (≈ 2.85–2.97); the differences are smaller than one standard deviation (≈ 0.5), so no major stands out.
* `credits` and `year` are positively correlated as expected; GPA is uncorrelated with both.

That is a complete, honest Stage 1 EDA. Open `02_exercises.ipynb` next, then `project/stage1_template.ipynb`.